# Runbook: Schema Drift Detection

## Purpose
Detect unexpected schema changes across source systems and data layers.

## When to Use
- Regular audits (monthly/quarterly)
- After source system upgrades
- When queries start failing with column errors
- Customer reports data type issues

## What This Checks
- Column additions/deletions
- Data type changes
- Nullability changes
- Schema consistency across layers

In [ ]:
# Setup
import os
import sys
os.environ['SPARK_HOME'] = '/opt/spark'
sys.path.insert(0, '/opt/spark/python')
sys.path.insert(0, '/opt/spark/python/lib/py4j-0.10.9.7-src.zip')

from pyspark.sql import SparkSession
import pandas as pd
from datetime import datetime

spark = SparkSession.builder \
    .appName("SchemaDriftDetection") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print(f"✅ Schema Drift Detection Started: {datetime.now()}")

In [ ]:
# PARAMETERS
TABLE_NAME = "customers"  # Table to check
LAYERS = ["raw", "bronze", "silver", "gold"]  # Layers to audit

## Check 1: Get Current Schema for All Layers

In [ ]:
def get_schema_info(layer, table):
    """Get schema information for a table"""
    try:
        schema_df = spark.sql(f"DESCRIBE nessie.{layer}.{table}")
        schema_list = [(row['col_name'], row['data_type']) for row in schema_df.collect()]
        return schema_list
    except Exception as e:
        return None

# Get schemas for all layers
schemas = {}
for layer in LAYERS:
    schemas[layer] = get_schema_info(layer, TABLE_NAME)
    if schemas[layer]:
        print(f"\n=== {layer.upper()} LAYER SCHEMA ===")
        for col_name, data_type in schemas[layer]:
            print(f"  {col_name}: {data_type}")
    else:
        print(f"\n⚠️  {layer.upper()} layer table not found")

## Check 2: Compare Schemas Across Layers

In [ ]:
# Compare raw vs bronze vs silver vs gold
print("\n=== SCHEMA DRIFT ANALYSIS ===")

for i in range(len(LAYERS) - 1):
    layer1 = LAYERS[i]
    layer2 = LAYERS[i + 1]
    
    if not schemas[layer1] or not schemas[layer2]:
        continue
    
    schema1_dict = dict(schemas[layer1])
    schema2_dict = dict(schemas[layer2])
    
    # Find differences
    added_cols = set(schema2_dict.keys()) - set(schema1_dict.keys())
    removed_cols = set(schema1_dict.keys()) - set(schema2_dict.keys())
    
    # Check type changes
    type_changes = []
    for col in set(schema1_dict.keys()) & set(schema2_dict.keys()):
        if schema1_dict[col] != schema2_dict[col]:
            type_changes.append((col, schema1_dict[col], schema2_dict[col]))
    
    print(f"\n{layer1.upper()} → {layer2.upper()}:")
    
    if added_cols:
        print(f"  ➕ Columns added: {', '.join(added_cols)}")
    if removed_cols:
        print(f"  ➖ Columns removed: {', '.join(removed_cols)}")
    if type_changes:
        print(f"  🔄 Type changes:")
        for col, old_type, new_type in type_changes:
            print(f"     {col}: {old_type} → {new_type}")
    if not added_cols and not removed_cols and not type_changes:
        print(f"  ✅ Schemas match")

## Check 3: Review Table History for Schema Changes

In [ ]:
# Check when schema last changed
print("\n=== SCHEMA CHANGE HISTORY ===")

for layer in LAYERS:
    if not schemas[layer]:
        continue
    
    try:
        history = spark.sql(f"SELECT * FROM nessie.{layer}.{TABLE_NAME}.history LIMIT 10")
        print(f"\n{layer.upper()} - Recent changes:")
        history.select("made_current_at", "snapshot_id", "is_current_ancestor").show(truncate=False)
    except Exception as e:
        print(f"  Could not retrieve history: {e}")

## Check 4: List All Tables Across Layers

In [ ]:
# Get list of all tables in each layer
print("\n=== TABLE INVENTORY ===")

all_tables = {}
for layer in LAYERS:
    try:
        tables = spark.sql(f"SHOW TABLES IN nessie.{layer}")
        table_list = [row['tableName'] for row in tables.collect()]
        all_tables[layer] = set(table_list)
        print(f"\n{layer.upper()}: {len(table_list)} tables")
        print(f"  {', '.join(sorted(table_list))}")
    except Exception as e:
        print(f"\n{layer.upper()}: Error - {e}")
        all_tables[layer] = set()

# Check for missing tables across layers
print("\n=== MISSING TABLES ===")
for i in range(len(LAYERS) - 1):
    layer1 = LAYERS[i]
    layer2 = LAYERS[i + 1]
    missing = all_tables[layer1] - all_tables[layer2]
    if missing:
        print(f"⚠️  In {layer1.upper()} but not in {layer2.upper()}: {', '.join(missing)}")
    else:
        print(f"✅ All {layer1.upper()} tables exist in {layer2.upper()}")

## Summary & Recommendations

### Findings:
- 

### Actions Required:
- [ ] Update dbt models if schema changes detected
- [ ] Notify affected customers
- [ ] Update documentation
- [ ] Review transformation logic

### Follow-up:
- Next audit date: 
- Assigned to: 